# Independent pretrained DiffJev adaptations

Formal DiffJev workflow; see AGENTS.md and prompt.md.

In [ ]:
import os, sys, gc
from pathlib import Path
ROOT = Path(os.environ.get('DIFFJEV_ROOT', Path.cwd())).resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'AGENTS.md').exists() and Path('/content/drive/MyDrive/DiffJev/AGENTS.md').exists():
    ROOT = Path('/content/drive/MyDrive/DiffJev')
assert (ROOT / 'AGENTS.md').exists(), 'Mount Drive and set DIFFJEV_ROOT to the checkout used by notebook 00'
sys.path.insert(0, str(ROOT / 'src'))
from diffjev.notebook_runtime import NotebookRuntime
runtime = NotebookRuntime(ROOT)
runtime.python_code("from diffjev.preparation import check_environment; from pathlib import Path; check_environment(Path.cwd())", label='02_train_diffjev_environment')
def run_python(source):
    prefix = '''import gc
from pathlib import Path
from diffjev import workflows as w
from diffjev.config import load_yaml_config
ROOT = w.ROOT
cfg = load_yaml_config(str(ROOT / 'configs/workflow.yaml'))
SEED, DEVICE = cfg['seed'], cfg['device']
'''
    runtime.python_code(prefix + source, label='02_train_diffjev')
print('Experiment worker:', runtime.python)


These cells perform real supervised training. The saved source/augmentation trace, 3,125 updates, final-four-layer-plus-normalization policy, development checkpoint rule, and separate calibration split are shared. Set RUNS to the machines/runs allocated to this session. Three anchor seeds are planned; the size panel initially uses seed 0.

In [ ]:
run_python(r"""
RUNS = cfg['backbones']
for backbone in RUNS:
    summary = w.train_run(backbone, seed=SEED, device=DEVICE)
    print(backbone, summary)
    gc.collect()
""")

Resume a specific interrupted adaptation with w.train_run(..., resume=".../step_500.pt"). CPU/CUDA/Python/NumPy RNG, corruption generator, augmentation generator, counters, and trace identity are restored.